## **Default Chat Model Setup**

In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

google_model="gemini-3.1-flash-lite"

## **Fundamental Process**

In [2]:
# Task 1 : Prompt Generation with Dynamic Inputs
from langchain_core.prompts import ChatPromptTemplate

dynamic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a genz assistant. You are very chill and use a lot of slang. You are also very funny and sarcastic."),
    ("user", "Write a post about {topic}.")
])

In [3]:
# Task 2 : LLM
from langchain.chat_models import init_chat_model

llm_gemini = init_chat_model(model=google_model,model_provider="openai",
    openai_api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [4]:
# Task 3 : String Output Parser
from langchain_core.output_parsers import StrOutputParser

str_output_parser = StrOutputParser()

In [5]:
# Task 4 : Custom Runnable
from langchain_core.runnables import RunnableLambda

def custom_runnable(input_str: str) -> dict:
    # Custom logic to process the input string and return a dictionary
    return {"content": input_str}

custom_runnable_instance = RunnableLambda(custom_runnable)

In [6]:
## **Parallel Chain 1**

In [7]:

instagram_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a instagram content creator. You are very chill and use a lot of slang. You are also very funny and sarcastic."),
    ("user", "Give me a readymade Instagram post: {content}. Make sure to include hashtags and emojis. Only give me the post, do not include any other text.")
])

instagram_chain = instagram_prompt | llm_gemini | str_output_parser


## **Parallel Chain 2**

In [8]:
def generate_linkedin_post(content_dict: dict) -> str:
    content = content_dict.get("content", "")
    linkedin_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn content creator. You are very professional and use a lot of industry-specific terminology."),
        ("user", "Give me a readymade LinkedIn post: {content}. Make sure to include a professional tone and relevant hashtags. Only give me the post, do not include any other text.")
    ])

    linkedin_chain = linkedin_prompt | llm_gemini | str_output_parser

    response = linkedin_chain.invoke({"content": content})

    return response

## **Custom Parallel Chain**

In [9]:
from langchain_core.runnables import RunnableParallel
parallel_chain =  dynamic_prompt | llm_gemini | str_output_parser | custom_runnable_instance | RunnableParallel(branches={
    "instagram": instagram_chain,
    "linkedin": RunnableLambda(generate_linkedin_post)
})

In [10]:
parallel_chain.invoke({"topic": input("Enter a topic for the post: ")})

{'branches': {'instagram': 'Okay, so let’s talk about cars for a sec. 🚗💨\n\nWhy are we all out here acting like a Honda Civic is a personality trait? Like, calm down, bestie. It’s a box on wheels that gets you to Taco Bell at 2 AM, not a spaceship. \n\nAnd don\'t even get me started on the car bros. You know the ones. They spend their entire paycheck on a spoiler that adds, like, 0.5 horsepower and makes their car sound like a lawnmower fighting for its life in a blender. We get it, Kyle, your exhaust is loud. Everyone in a three-mile radius is definitely impressed and not at all annoyed by your little "vroom vroom" energy. 🙄\n\nAlso, can we address the people who don’t know how to use a blinker? It’s literally one move with your hand. It’s not that deep, but yet, here we are, playing real-life Mario Kart every time we get on the highway. \n\nAnyway, if your car has an Aux cord and gets me from point A to point B without exploding, you’re doing great sweetie. Everything else is just ex

## **Chain as a Runnable**

In [11]:
def beautify_output(output_dict: dict) -> dict:
    instagram_post = output_dict["branches"].get("instagram", "")
    linkedin_post = output_dict["branches"].get("linkedin", "")

    return {"Instagram Post": instagram_post, "LinkedIn Post": linkedin_post}

beautified_runnable = RunnableLambda(beautify_output)

In [12]:
final_chain = parallel_chain | beautified_runnable

In [13]:
final_chain.invoke({"topic": input("Enter a topic for the post: ")})

{'Instagram Post': 'Listen, can we talk about how people treat their cars like they’re their actual children? Bestie, it’s a hunk of metal that depreciates the second you drive it off the lot. Relax. 💀\n\nI’m seeing dudes out here spending their entire paycheck on "modifications" just so their Honda Civic can sound like a lawnmower having a panic attack at 3 AM. Like, congrats, Tyler? Everyone in a five-mile radius now knows you have a fragile ego. Truly, a cinematic masterpiece. 🏎️💨\n\nAnd don’t even get me started on the "car community" meetups in random parking lots. You’re just standing around in the cold drinking lukewarm gas station coffee staring at someone else’s engine bay. The vibes are officially rancid. \n\nLike, I get it, you love your manual transmission. Congrats on having to work harder to get to Taco Bell. That’s so brave of you. 💅✨\n\nAnyway, if your car doesn’t have an aux cord or Bluetooth that actually connects on the first try, just throw the whole thing in the tr